# 日本語の文章の校正すべき箇所を指摘する（表記ゆれ・冗長な表現・ら抜き言葉）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/25_gec_education/proofreading_ja.ipynb)

NLP クックブック 25-11 ／ 文法誤り訂正と言語教育

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

報告書やメールの下書きから、表記ゆれ、冗長な言い回し、ら抜き言葉、二重敬語、長すぎる文、同じ助詞の連続を見つけて、書き直しの案と一緒に指摘したい。形態素解析による規則、textlint、手元の LLM の 3 つを比べる。

In [1]:
!pip install -q fugashi unidic-lite
# textlint (MIT) は Node.js の校正ツール。Colab には Node.js が入っているので、作業ディレクトリに入れて Python から呼ぶ
!npm install --silent --no-audit --no-fund textlint@15.8.0 textlint-rule-preset-ja-technical-writing@12.0.2 @textlint-ja/textlint-rule-no-synonyms@1.3.0 sudachi-synonyms-dictionary@20.0.0

In [2]:
import re
from collections import defaultdict
from functools import lru_cache
from fugashi import Tagger

tagger = Tagger()

def split_sentences(text):
    return [s for s in re.split(r"(?<=[。！？])", text) if s.strip()]

REDUNDANT = [(r"することができ", "「できる」で足りる"), (r"することが可能", "「できる」で足りる"),
             (r"まず最初に|一番最初", "「まず」か「最初に」のどちらかにする"), (r"であると言え", "「である」で足りる"),
             (r"(?<!を)を行[いうえっわ]", "「〜する」にする")]
HONORIFIC = {"仰る", "為さる", "いらっしゃる", "召し上がる", "下さる"}     # UniDic の語彙素 (lemma) で書く

@lru_cache(maxsize=None)     # 同じ語を何度も解析しないように結果を覚えておく
def is_ichidan(lemma):
    """一段動詞かカ変動詞か (ら抜きが起きるのはこれだけ。「帰れる」のような五段動詞の可能形は正しい)"""
    ws = tagger(lemma)
    return len(ws) == 1 and ws[0].feature.cType.startswith(("上一段", "下一段", "カ行変格"))

def rule_check(text):
    """(種類, 箇所, 書き直しの案) のリストを返す"""
    issues = []
    for s in split_sentences(text):
        ws = list(tagger(s))
        for pat, advice in REDUNDANT:
            issues += [("冗長", m.group(), advice) for m in re.finditer(pat, s)]
        for i, w in enumerate(ws):
            f = w.feature
            # ら抜き (1): UniDic は「起きれる」「見れる」を 1 語 (書字形 起きれる、語彙素 起きる) として登録している
            if f.pos1 == "動詞" and f.lemma and f.orthBase == f.lemma[:-1] + "れる" and f.lemma.endswith("る") and is_ichidan(f.lemma):
                issues.append(("ら抜き", w.surface, f"「{f.lemma[:-1]}られる」の形にする"))
            # ら抜き (2): 「食べ」(一段の未然形) + 助動詞「れる」と解析される場合
            if f.pos1 == "動詞" and f.cForm.startswith("未然") and f.cType.startswith(("上一段", "下一段", "カ行変格")) \
                    and i + 1 < len(ws) and ws[i + 1].feature.lemma == "れる":
                issues.append(("ら抜き", w.surface + ws[i + 1].surface, f"「{f.lemma[:-1]}られる」の形にする"))
            # 二重敬語: 尊敬語 (おっしゃる、召し上がる、お〜になる、ご覧になる) に尊敬の「れる・られる」を重ねる
            if f.pos1 == "助動詞" and f.lemma in ("れる", "られる") and i > 0:
                prev = ws[i - 1].feature
                start = None                       # 尊敬語の始まりの位置 (指摘する箇所の先頭)
                if prev.lemma in HONORIFIC:
                    start = i - 1
                elif prev.lemma == "成る" and i >= 3 and ws[i - 2].surface == "に":
                    # 「ご覧になる」か「お/ご + 語 + になる」
                    starts = [j for j in range(max(0, i - 6), i - 2)
                              if ws[j].feature.lemma == "御覧" or (ws[j].feature.pos1 == "接頭辞" and ws[j].surface in ("お", "ご"))]
                    start = starts[-1] if starts else None
                if start is not None:
                    issues.append(("二重敬語", "".join(x.surface for x in ws[start:i + 1]), "「れる・られる」を取る"))
        # 長すぎる文: 80 字を超えるか、読点が 4 つ以上
        if len(s) > 80 or s.count("、") >= 4:
            issues.append(("長文", s[:12] + "…", f"{len(s)} 字・読点 {s.count('、')} 個。2 文に分ける"))
        # 助詞の連続: 読点で区切った範囲に同じ助詞 (が・を・に・で・は) が 2 回、または格助詞の「の」が 3 回以上
        for clause in s.split("、"):
            ps = [w for w in tagger(clause) if w.feature.pos1 == "助詞"]
            for p in ("が", "を", "に", "で", "は"):
                if sum(w.surface == p for w in ps) >= 2:
                    issues.append(("助詞連続", clause[:12] + "…", f"「{p}」が 2 回。語順や言い方を変える"))
            if sum(w.surface == "の" and w.feature.pos2 == "格助詞" for w in ps) >= 3:
                issues.append(("助詞連続", clause[:12] + "…", "「の」が 3 回以上。言い換える"))
    # 表記ゆれ: 文書の中で、同じ語彙素 (lemma) が違う書字形 (orthBase) で書かれている
    forms = defaultdict(set)
    for w in tagger(text):
        if w.feature.pos1 in ("名詞", "動詞", "形容詞", "副詞") and w.feature.lemma:
            forms[(w.feature.lemma, w.feature.pos1)].add(w.feature.orthBase or w.surface)
    issues += [("表記ゆれ", "/".join(sorted(fs)), "どれかにそろえる") for fs in forms.values() if len(fs) > 1]
    return issues

# 動作確認用に手作りした文書 (1〜2 文) と、含まれる問題の種類 (正解)。空集合は「問題なし」
DOCS = [
    ("このツールを使えば、作業を自動化することができます。", {"冗長"}),
    ("会議では、まず最初に前回の議事録を確認します。", {"冗長"}),
    ("この問題は、設定を変えるだけで解決することが可能です。", {"冗長"}),
    ("各部署ごとに担当者を一人決めてください。", {"冗長"}),
    ("頭痛が痛いので、今日は早めに帰ります。", {"冗長"}),
    ("この件について、来週検討を行いたいと思います。", {"冗長"}),
    ("後で後悔しないように、よく確認しよう。", {"冗長"}),
    ("朝は6時に起きれるようになった。", {"ら抜き"}),
    ("明日のパーティーには来れますか。", {"ら抜き"}),
    ("この店では新鮮な刺身が食べれる。", {"ら抜き"}),
    ("その映画はもう見れない。", {"ら抜き"}),
    ("虫が入るので窓を開けれない。", {"ら抜き"}),
    ("社長がそのようにおっしゃられました。", {"二重敬語"}),
    ("先生はもう資料をご覧になられましたか。", {"二重敬語"}),
    ("お客様が料理を召し上がられています。", {"二重敬語"}),
    ("ご注文の品はお揃いになられましたでしょうか。", {"二重敬語"}),
    ("部長が資料をお読みになられた。", {"二重敬語"}),
    ("子供の数が減っている。子どもの遊び場も少なくなった。", {"表記ゆれ"}),
    ("結果が分かったら連絡します。わからない点は聞いてください。", {"表記ゆれ"}),
    ("設定ファイルをサーバに置く。次にサーバーを再起動する。", {"表記ゆれ"}),
    ("申込書を出してください。締め切りを過ぎた申し込みは受け付けません。", {"表記ゆれ"}),
    ("ウェブサイトを更新した。Webサイトのアドレスは変わらない。", {"表記ゆれ"}),
    ("打ち合わせは明日です。打合せの資料を準備します。", {"表記ゆれ"}),
    ("私の兄の友人の会社の社長に会った。", {"助詞連続"}),
    ("この案は予算は少ないが効果は大きい。", {"助詞連続"}),
    ("データの分析の結果の報告は来週です。", {"助詞連続"}),
    ("本研究では近年急速に普及している大規模言語モデルを用いて日本語の学習者が書いた作文に含まれる文法的な誤りを自動的に検出し"
     "その誤りの種類に応じた説明を生成する手法を提案し実際の作文による評価実験で有効性を検証した。", {"長文"}),
    ("今回の改修では、画面の配色、ボタンの配置、入力欄の大きさ、エラーメッセージの文言、ヘルプの内容を見直した。", {"長文"}),
    ("次回の打ち合わせでは昨年度の売上実績と今年度の目標を比較したうえで来期に向けた営業戦略の方向性について"
     "各担当者から意見を出してもらい最終的な方針を決める予定です。", {"長文"}),
    ("先生がおっしゃられたとおり、この方法で問題を解決することができます。", {"二重敬語", "冗長"}),
    ("明日は6時に起きれるので、子どもを駅まで送る。子供はきっと喜ぶだろう。", {"ら抜き", "表記ゆれ"}),
    # 以下は問題のない文 (規則が誤検出しやすい形を含む)
    ("この服はまだ着られる。", set()), ("駅から歩いて帰れる距離だ。", set()), ("この道は狭くて車が通れない。", set()),
    ("この部屋からは富士山が見られる。", set()), ("部長は明日の会議にいらっしゃいます。", set()), ("先生がおっしゃったとおりです。", set()),
    ("東京に住んでいる友人に手紙を送った。", set()), ("彼は私が書いた本が好きだ。", set()), ("忙しいときは昼食を抜くことがある。", set()),
    ("新しい制度は来月から始まるが、詳しい手続きは担当者から改めて連絡する予定である。", set()),
    ("この方法なら、誰でも簡単に設定できます。", set()), ("会議は午後3時から始まります。資料は事前に配布します。", set()),
]
print(len(DOCS), "文書")
rule_pred = []
for text, gold in DOCS:
    issues = rule_check(text)
    rule_pred.append({c for c, _, _ in issues})
    print(f"{'○' if rule_pred[-1] == gold else '×'} {text[:24]}  正解 {sorted(gold)}")
    for c, span, advice in issues:
        print(f"      {c}: {span} → {advice}")

43 文書
○ このツールを使えば、作業を自動化することができま  正解 ['冗長']
      冗長: することができ → 「できる」で足りる
○ 会議では、まず最初に前回の議事録を確認します。  正解 ['冗長']
      冗長: まず最初に → 「まず」か「最初に」のどちらかにする
○ この問題は、設定を変えるだけで解決することが可能  正解 ['冗長']
      冗長: することが可能 → 「できる」で足りる
× 各部署ごとに担当者を一人決めてください。  正解 ['冗長']
× 頭痛が痛いので、今日は早めに帰ります。  正解 ['冗長']
○ この件について、来週検討を行いたいと思います。  正解 ['冗長']
      冗長: を行い → 「〜する」にする
× 後で後悔しないように、よく確認しよう。  正解 ['冗長']
○ 朝は6時に起きれるようになった。  正解 ['ら抜き']
      ら抜き: 起きれる → 「起きられる」の形にする
○ 明日のパーティーには来れますか。  正解 ['ら抜き']
      ら抜き: 来れ → 「来られる」の形にする
○ この店では新鮮な刺身が食べれる。  正解 ['ら抜き']
      ら抜き: 食べれる → 「食べられる」の形にする
○ その映画はもう見れない。  正解 ['ら抜き']
      ら抜き: 見れ → 「見られる」の形にする
○ 虫が入るので窓を開けれない。  正解 ['ら抜き']
      ら抜き: 開けれ → 「開けられる」の形にする
○ 社長がそのようにおっしゃられました。  正解 ['二重敬語']
      二重敬語: おっしゃられ → 「れる・られる」を取る
○ 先生はもう資料をご覧になられましたか。  正解 ['二重敬語']
      二重敬語: ご覧になられ → 「れる・られる」を取る
○ お客様が料理を召し上がられています。  正解 ['二重敬語']
      二重敬語: 召し上がられ → 「れる・られる」を取る
○ ご注文の品はお揃いになられましたでしょうか。  正解 ['二重敬語']
      二重敬語: お揃いになられ → 「れる・られる」を取る
○ 部長が資料をお読みになられた。  正解 ['二重敬語']
      二重敬語: お読みになられ 

In [3]:
# textlint を Python から呼ぶ。文書ごとにファイルに書き、まとめて 1 回で検査して JSON で受け取る
import json, os, subprocess
from pathlib import Path

config = {"rules": {"preset-ja-technical-writing": {"sentence-length": {"max": 80}},   # 文の長さの上限を規則とそろえる (既定は 100 字)
                    "@textlint-ja/no-synonyms": True}}
Path(".textlintrc.json").write_text(json.dumps(config))
Path("docs").mkdir(exist_ok=True)
paths = []
for i, (text, _) in enumerate(DOCS):
    paths.append(f"docs/{i:02d}.txt")
    Path(paths[-1]).write_text(text + "\n")
r = subprocess.run(["npx", "textlint", "--format", "json", *paths], capture_output=True, text=True)
out = r.stdout
if not out:      # インストールの失敗などで textlint が動かなかったときは、エラーを表示する
    print(r.stderr)

# 規則の名前を、この表の種類に対応づける (二重敬語の規則はプリセットにない。ほかの規則の指摘は「その他」)
TL_CAT = {"ja-no-redundant-expression": "冗長", "no-dropping-the-ra": "ら抜き", "no-doubled-joshi": "助詞連続",
          "sentence-length": "長文", "max-ten": "長文", "no-synonyms": "表記ゆれ"}
tl_msgs = {p: [] for p in paths}
for f in json.loads(out):
    tl_msgs[os.path.relpath(f["filePath"])] = [(m["ruleId"].split("/")[-1], m["message"].split("\n")[0]) for m in f["messages"]]
tl_pred = [{TL_CAT[r] for r, _ in tl_msgs[p] if r in TL_CAT} for p in paths]
for (text, gold), p in zip(DOCS, paths):
    if tl_msgs[p]:
        print(text[:24], f"正解 {sorted(gold)}")
        for r, m in tl_msgs[p]:
            print(f"      [{TL_CAT.get(r, 'その他')}] {r}: {m[:60]}")

このツールを使えば、作業を自動化することができま 正解 ['冗長']
      [冗長] ja-no-redundant-expression: 【dict2】 "することができます"は冗長な表現です。"することが"を省き簡潔な表現にすると文章が明瞭になります。
この問題は、設定を変えるだけで解決することが可能 正解 ['冗長']
      [冗長] ja-no-redundant-expression: 【dict1】 "することが可能です"は冗長な表現です。"することが"を省き簡潔な表現にすると文章が明瞭になります。
この件について、来週検討を行いたいと思います。 正解 ['冗長']
      [冗長] ja-no-redundant-expression: 【dict5】 "検討を行う"は冗長な表現です。"検討する"など簡潔な表現にすると文章が明瞭になります。
      [その他] ja-no-weak-phrase: 弱い表現: "思います" が使われています。
朝は6時に起きれるようになった。 正解 ['ら抜き']
      [ら抜き] no-dropping-the-ra: ら抜き言葉を使用しています。
      [助詞連続] no-doubled-joshi: 一文に二回以上利用されている助詞 "に" がみつかりました。
明日のパーティーには来れますか。 正解 ['ら抜き']
      [ら抜き] no-dropping-the-ra: ら抜き言葉を使用しています。
この店では新鮮な刺身が食べれる。 正解 ['ら抜き']
      [ら抜き] no-dropping-the-ra: ら抜き言葉を使用しています。
その映画はもう見れない。 正解 ['ら抜き']
      [ら抜き] no-dropping-the-ra: ら抜き言葉を使用しています。
虫が入るので窓を開けれない。 正解 ['ら抜き']
      [ら抜き] no-dropping-the-ra: ら抜き言葉を使用しています。
設定ファイルをサーバに置く。次にサーバーを再起動 正解 ['表記ゆれ']
      [表記ゆれ] no-synonyms: 同義語である「サーバ」と「サーバー」が利用されています
この案は予算は少ないが効果は大きい。 正解 ['助詞連続']
 

In [4]:
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"     # MIT ライセンス
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる (注意書きを参照)。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

@torch.no_grad()
def chat(prompt, max_new_tokens=200):
    ids = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt", return_dict=True).to(llm.device)
    out = llm.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.05, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

CATS = ["表記ゆれ", "冗長", "ら抜き", "二重敬語", "長文", "助詞連続"]
PROMPT = """あなたは日本語の文章の校正者です。次の文章に、下の 6 種類の問題があれば指摘してください。
- 表記ゆれ: 同じ語が違う書き方で書かれている (例: 子供と子ども)
- 冗長: 削っても意味が変わらない言い回し (例: することができる → できる)
- ら抜き: 「見られる」を「見れる」とするような、可能の「られる」の「ら」が抜けた形
- 二重敬語: 敬語を重ねた形 (例: おっしゃられる → おっしゃる)
- 長文: 長すぎて読みにくい文
- 助詞連続: 同じ助詞が 1 つの文の中で続いている (例: 私の兄の友人の会社)
問題 1 つにつき 1 行で「種類|問題の箇所|書き直しの案」の形で出力してください。問題がなければ「なし」とだけ出力してください。説明は書かないでください。

文章: {}"""

def llm_check(text):
    """「種類|箇所|案」の行だけを取り出す。箇所が「なし」の行や、後に続く説明の文は捨てる"""
    issues = []
    for line in chat(PROMPT.format(text)).splitlines():
        f = [x.strip() for x in line.split("|")]
        if len(f) >= 2 and f[0] in CATS and f[1] not in ("", "なし"):
            issues.append((f[0], f[1], f[2] if len(f) > 2 else ""))
    return issues

# 43 文書の生成は T4 で 3 分ほど (初回はモデルのダウンロードに数分かかる)
llm_pred = []
for text, gold in DOCS:
    issues = llm_check(text)
    llm_pred.append({c for c, _, _ in issues})
    print(f"{'○' if llm_pred[-1] == gold else '×'} {text[:24]}  正解 {sorted(gold)}")
    for c, span, advice in issues:
        print(f"      {c}: {span[:30]} → {advice[:40]}")

× このツールを使えば、作業を自動化することができま  正解 ['冗長']
× 会議では、まず最初に前回の議事録を確認します。  正解 ['冗長']
× この問題は、設定を変えるだけで解決することが可能  正解 ['冗長']
      表記ゆれ: 「変える」と「換える」 → 「変える」に統一
      冗長: 「だけで」 → 「のみで」または「単に」
× 各部署ごとに担当者を一人決めてください。  正解 ['冗長']
      表記ゆれ: 各部署ごとに→各部署に → 各部署に担当者を一人決めてください。
× 頭痛が痛いので、今日は早めに帰ります。  正解 ['冗長']
      表記ゆれ: 「痛い」が同じ文の中で違う漢字で書かれている (「痛い」と「 → 「頭痛が痛むので、今日は早めに帰ります。」または「頭が痛いので、今日は早めに帰り
      冗長: 「痛い」が重複している → 「頭痛があるので、今日は早めに帰ります。」
      助詞連続: 「ので」が連続して使われている → 「頭痛があるので、今日は早めに帰ります。」(または文脈に応じて「のため」「ために
      長文: やや長文ですが、読みにくいほどではない → 必要に応じて、句読点の追加や段落分けを行うことで改善できます。例えば、「頭痛が痛
× この件について、来週検討を行いたいと思います。  正解 ['冗長']
× 後で後悔しないように、よく確認しよう。  正解 ['冗長']
○ 朝は6時に起きれるようになった。  正解 ['ら抜き']
      ら抜き: 起きれる→起きられる → 起きられるようになった。
× 明日のパーティーには来れますか。  正解 ['ら抜き']
○ この店では新鮮な刺身が食べれる。  正解 ['ら抜き']
      ら抜き: 「食べれる」 → 「食べられる」
○ その映画はもう見れない。  正解 ['ら抜き']
      ら抜き: その映画はもう見れない → その映画はもう見られない。
○ 虫が入るので窓を開けれない。  正解 ['ら抜き']
      ら抜き: 虫が入るので窓を開けれない → 虫が入るので窓を開けられない。
○ 社長がそのようにおっしゃられました。  正解 ['二重敬語']
      二重敬語: 「おっしゃられました」 → おっしゃる
○

In [5]:
# 文書 × 種類の単位で、指摘の適合率・再現率を比べる
import pandas as pd

def prf(preds, cat):
    tp = sum(cat in p and cat in g for p, (_, g) in zip(preds, DOCS))
    fp = sum(cat in p and cat not in g for p, (_, g) in zip(preds, DOCS))
    fn = sum(cat not in p and cat in g for p, (_, g) in zip(preds, DOCS))
    return f"{tp}/{tp + fp}" if tp + fp else "-", f"{tp}/{tp + fn}"

COLS = ("適合 (正しい指摘/指摘)", "再現 (見つけた/正解)")
table = {sys_name: {(cat, col): v for cat in CATS for col, v in zip(COLS, prf(preds, cat))}
         for sys_name, preds in [("規則", rule_pred), ("textlint", tl_pred), ("LLM", llm_pred)]}
pd.DataFrame(table)

規則 textlint   LLM
表記ゆれ 適合 (正しい指摘/指摘)  6/6      2/2  7/19
     再現 (見つけた/正解)   6/7      2/7   7/7
冗長   適合 (正しい指摘/指摘)  5/5      4/4   2/4
     再現 (見つけた/正解)   5/8      4/8   2/8
ら抜き  適合 (正しい指摘/指摘)  6/6      6/6   5/6
     再現 (見つけた/正解)   6/6      6/6   5/6
二重敬語 適合 (正しい指摘/指摘)  6/6        -   4/4
     再現 (見つけた/正解)   6/6      0/6   4/6
長文   適合 (正しい指摘/指摘)  3/3      2/2   2/3
     再現 (見つけた/正解)   3/3      2/3   2/3
助詞連続 適合 (正しい指摘/指摘)  3/7      1/3   0/7
     再現 (見つけた/正解)   3/3      1/3   0/3

In [6]:
# 誤検出 (正解にない指摘) と見逃し (正解にあるのに指摘なし) を並べる
for sys_name, preds in [("規則", rule_pred), ("textlint", tl_pred), ("LLM", llm_pred)]:
    print(f"■ {sys_name}")
    for p, (text, gold) in zip(preds, DOCS):
        if p - gold:
            print(f"   誤検出 {sorted(p - gold)}: {text[:30]}")
        if gold - p:
            print(f"   見逃し {sorted(gold - p)}: {text[:30]}")

■ 規則
   見逃し ['冗長']: 各部署ごとに担当者を一人決めてください。
   見逃し ['冗長']: 頭痛が痛いので、今日は早めに帰ります。
   見逃し ['冗長']: 後で後悔しないように、よく確認しよう。
   見逃し ['表記ゆれ']: ウェブサイトを更新した。Webサイトのアドレスは変わらない。
   誤検出 ['助詞連続']: 本研究では近年急速に普及している大規模言語モデルを用いて日本
   誤検出 ['助詞連続']: 次回の打ち合わせでは昨年度の売上実績と今年度の目標を比較した
   誤検出 ['助詞連続']: 東京に住んでいる友人に手紙を送った。
   誤検出 ['助詞連続']: 彼は私が書いた本が好きだ。
■ textlint
   見逃し ['冗長']: 会議では、まず最初に前回の議事録を確認します。
   見逃し ['冗長']: 各部署ごとに担当者を一人決めてください。
   見逃し ['冗長']: 頭痛が痛いので、今日は早めに帰ります。
   見逃し ['冗長']: 後で後悔しないように、よく確認しよう。
   誤検出 ['助詞連続']: 朝は6時に起きれるようになった。
   見逃し ['二重敬語']: 社長がそのようにおっしゃられました。
   見逃し ['二重敬語']: 先生はもう資料をご覧になられましたか。
   見逃し ['二重敬語']: お客様が料理を召し上がられています。
   見逃し ['二重敬語']: ご注文の品はお揃いになられましたでしょうか。
   見逃し ['二重敬語']: 部長が資料をお読みになられた。
   見逃し ['表記ゆれ']: 子供の数が減っている。子どもの遊び場も少なくなった。
   見逃し ['表記ゆれ']: 結果が分かったら連絡します。わからない点は聞いてください。
   見逃し ['表記ゆれ']: 申込書を出してください。締め切りを過ぎた申し込みは受け付けま
   見逃し ['表記ゆれ']: ウェブサイトを更新した。Webサイトのアドレスは変わらない。
   見逃し ['表記ゆれ']: 打ち合わせは明日です。打合せの資料を準備します。
   見逃し ['助詞連続']: 私の兄の友人の会社の社長に会った。
   見逃し ['助詞連続']: データの分析の結果の報告は来週です。
   見逃し

- 文法誤り訂正（この章の「LLM で日本語の文法誤りを訂正して評価する」など）は、誤りを含む文を正しい文 1 つに書き換える。校正支援は、文法的には正しくても読みにくい箇所に印を付けて理由と書き直しの案を示し、直すかどうかは書き手に任せる。そのため評価も、参照文との一致ではなく、指摘の正しさ（適合率）と見逃しの少なさ（再現率）で見る。
- 規則は、ら抜き・二重敬語・長文のように形で決まる問題に強い。UniDic は「起きれる」を語彙素「起きる」の別の書字形として登録しているので、語彙素と書字形を比べればら抜きも表記ゆれも見つかる。一方、パターンにない言い回し（「各部署ごと」「頭痛が痛い」「後で後悔」）は見逃し、助詞連続は 7 文書で指摘して正しいのは 3 文書だった（「東京に住んでいる友人に」のような自然な重なりや、読点のない長い文を誤検出）。規則と例文を同時に作ったので、規則の成績は実際より良く見える。
- 語彙素の違う「ウェブサイト/Webサイト」は、規則でも textlint でも見つからなかった（LLM は見つけた）。日本語の形態素解析と文分割の章の「Sudachi の正規化形で表記ゆれを吸収する」の正規化形を使えば拾える可能性があるが、ここでは確かめていない。
- textlint は、規則をプラグインとして組み合わせるテキストの校正ツール (Node.js) で、日本語の規則が多く公開されている。ここでは技術文書向けのプリセットと、Sudachi 同義語辞書を使う表記ゆれの規則を `subprocess` で呼んだ。冗長・ら抜き・長文・表記ゆれの指摘はすべて正しかったが、表記ゆれは 2/7 しか見つからず、二重敬語の規則はプリセットにない (0/6)。助詞連続は 3 文書で指摘して 2 文書が誤検出で、README のとおり連体化の「の」の重なりは許すので「私の兄の友人の会社の」は見逃した。
- LLM（Sarashina2.2 3B、T4 と同じ float16）は、ら抜き 5/6、二重敬語 4/6（指摘した 4 文書はすべて正しい）で、書き直しの案も出せた。表記ゆれは 7 文書すべてで指摘したが（規則 6/7、textlint 2/7）、正解にない 12 文書でも指摘した（「「案」と「案」」など）。冗長は 2/8 で、うち 1 文書は「だけで」を挙げた的外れな指摘。助詞連続は 0/3 で、正解にない 7 文書で指摘した。長文は 2/3 を見つけたが、書き直しの案は英語になるなど崩れていた。
- LLM の指摘は、種類が合っていても中身が誤っていることがある。表記ゆれとして「「結果」と「分かる」」を挙げたり、「受け付けません」をら抜きとして「受け付けられません」に直したり、敬語を落とす案（「お読みになられた」→「読んだ」）を出したりした。評価は文書単位の「その種類を指摘したか」で箇所までは照合していないので、これらも正しい指摘に数えている。種類ごとの正解は 3〜8 文書しかなく、1〜2 件の差は誤差の範囲。
- 実務では、規則（textlint や自作の規則）で機械的に検出できるものを先に潰し、LLM には意味の重複や言い換えの提案のような規則で書きにくい部分を任せる、という分担が現実的。LLM の指摘はそのまま採用せず、書き手が判断する候補として出す。
- ツールとモデルのライセンス: textlint と使った規則 (MIT)、Sudachi 同義語辞書（sudachi-synonyms-dictionary、Apache-2.0）、sbintuitions/sarashina2.2-3b-instruct-v0.1（SB Intuitions、MIT）。sarashina のトークナイザーを直すコードについては、大規模言語モデルの章の「オープンな LLM とチャットする（チャットテンプレート）」の注意書きを参照。